# 🧠 Xavier SLM Micro-Expert Fine-Tuning: ast_blast_expert
**Base Model:** `Qwen/Qwen2.5-Coder-0.5B-Instruct` | **Target:** Compact GGUF (Q4_K_M)

This notebook was generated by the Xavier Cognitive Engine CLI.
It consumes an anonymized training bundle (`train.jsonl` / `eval.jsonl`) sanitized under **Privacy Level P3**.

In [ ]:
# 1. Install optimized fine-tuning dependencies
!pip install -q --upgrade pip
!pip install -q torch transformers datasets peft accelerate bitsandbytes trl
# Install llama.cpp build tools for GGUF conversion
!git clone --depth 1 https://github.com/ggerganov/llama.cpp.git /tmp/llama_cpp
!pip install -q -r /tmp/llama_cpp/requirements.txt

In [ ]:
# 2. Mount Google Drive or upload local train.jsonl
from google.colab import files
import os

if not os.path.exists('train.jsonl'):
    print('Please upload train.jsonl exported from Xavier:')
    uploaded = files.upload()
else:
    print('Found existing train.jsonl in workspace.')

In [ ]:
# 3. Execute Xavier LoRA Fine-Tuning
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer

MODEL_ID = 'Qwen/Qwen2.5-Coder-0.5B-Instruct'
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True
)

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
)

model = get_peft_model(model, peft_config)
dataset = load_dataset('json', data_files={'train': 'train.jsonl'})['train']

def format_prompts(batch):
    texts = []
    for instr, inp, out in zip(batch['instruction'], batch.get('input_context', ['']*len(batch['instruction'])), batch['output']):
        text = f'<|im_start|>user\n{instr}\n{inp}<|im_end|>\n<|im_start|>assistant\n{out}<|im_end|>'
        texts.append(text)
    return {'text': texts}

dataset = dataset.map(format_prompts, batched=True)

args = TrainingArguments(
    output_dir='./results',
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy='no'
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=512,
    args=args
)
trainer.train()

In [ ]:
# 4. Merge LoRA weights & Convert to GGUF
model = model.merge_and_unload()
merged_dir = './merged_model'
model.save_pretrained(merged_dir)
tokenizer.save_pretrained(merged_dir)

output_gguf = 'ast_blast_expert.gguf'
# Use llama.cpp script to convert to GGUF
!python /tmp/llama_cpp/convert_hf_to_gguf.py ./merged_model --outfile {output_gguf} --outtype q4_k_m
print(f'GGUF export ready: {output_gguf}')
files.download(output_gguf)